In [124]:
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("Using:", device)

PyTorch: 2.14.1
MPS available: True
Using: mps


In [125]:
import numpy as np


def initial_state():
    board = np.zeros((3, 3), dtype=int)  # 提示：np.zeros((行数, 列数), dtype=int)
    current_player = 1
    return board, current_player


# board, player = initial_state()

# board[1, 1] = 1   # 把当前玩家的棋子放到中央
# player = -1            # 切换到另一位玩家

# for row, col in [(1, 1), (0, 0)]:
#     board[row, col] = player   # 哪个变量存着这一手的棋子值？
#     player = -player            # 怎样把当前值的正负号翻转？

# for row, col in [(1, 1), (1, 1)]:
#     if board[row, col] == 0:
#         board[row, col] = player
#         player = -player

In [126]:
def legal_actions(board):
    moves = list()
    for row in range(3):
        for col in range(3):
            if board[row, col] == 0:
                moves.append((row, col))
    return moves

In [127]:
def has_row_win(board, player):
    for row in range(3):
        if np.all(board[row, :] == player):
            return True
    return False

In [128]:
def has_col_win(board, player):
    for col in range(3):
        if np.all(board[:, col] == player):
            return True
    return False

In [129]:
# def has_diag_win(board, player):
#     for i in np.diag(board):
#         if i != player:
#             for i in np.diag(np.fliplr(board)):
#                 if i != player:
#                     return False
#     return True
def has_diag_win(board, player):
    main_win = np.all(np.diag(board) == player)
    anti_win = np.all(np.diag(np.fliplr(board)) == player)
    return main_win or anti_win

In [130]:
def has_won(board, player):
    row = has_row_win(board, player)
    col = has_col_win(board, player)
    diag = has_diag_win(board, player)
    return row or col or diag

In [131]:
def game_result(board, player):
    if has_won(board, -player) == True:
        return -player
    elif len(legal_actions(board)) == 0:
        return 0
    return None

In [132]:
def step(board, player, row, col):
    result = game_result(board, player)
    if result != None:
        return board, player
    if 0 <= row < 3 and 0 <= col < 3 and board[row, col] == 0:
        board[row, col] = player
        player = -player
    return board, player

In [133]:
import random


def random_action(board):
    moves = random.choice(legal_actions(board))
    return moves

In [134]:
def play_random_game():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [135]:
# 终局评分
def terminal_value(result, player):
    if result == 0:
        return 0
    elif result == player:
        return 1
    else:
        return -1

In [136]:
# 从当前局面出发，假设双方都走得最好，轮到行动的玩家最终能拿几分？当棋盘上有好几个空格时，这个函数就更有用了。它会试走每个合法位置，再考虑对手会怎样应对。每条走法一旦走到终局，你写的 terminal_value() 就能给它打分。
def minimax_value(board, player):
    result = game_result(board, player)
    if result is not None:
        return terminal_value(result, player)
    scores = []
    for row, col in legal_actions(board):  # 是否有问题，遍历怎么能称为“走得最好”
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)
    return max(scores)

In [137]:
def minimax_action(board, player):
    moves = legal_actions(board)
    scores = []
    for row, col in moves:
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)

    best_index = scores.index(max(scores))
    return moves[best_index]

In [138]:
def play_minimax_vs_random():
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == -1:
            row, col = minimax_action(board, player)  # O 用 Minimax
        else:
            row, col = random_action(board)  # X 随机选
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [139]:
def play_minimax_vs_minimax():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = minimax_action(board, player)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

MTCS

In [140]:
# 从给定棋盘下，对输入玩家来说，随机下能有什么分数
# 复制棋盘，让双方随机下到终局，返回起始玩家视角的得分，赢 +1、平 0、输 -1。
def rollout(board, player):
    board = board.copy()
    start_player = player

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    result = game_result(board, player)

    return terminal_value(result, start_player)

In [141]:
board99 = np.array(
    [
        [1, 0, 1],  # X . X
        [1, 1, -1],  # X X O
        [-1, -1, 0],  # O O .
    ]
)
N = 0
W = 0
for i in range(20):
    N = N + 1
    W = W + rollout(board99, -1)
Q = W / N  # 得分除局数，平均得分
print(N, W, Q)

20 6 0.3


In [142]:
# 对moves单独打印得分
moves99 = legal_actions(board99)
for row, col in moves99:
    child_board, next_player = step(board99.copy(), -1, row, col)
    N = 0
    W = 0
    for i in range(5):
        N = N + 1
        W = W - rollout(child_board, next_player)
    Q = W / N
    print((row, col), N, W, Q)

(0, 1) 5 -5 -1.0
(2, 2) 5 5 1.0


In [143]:
child_board, next_player = step(board99.copy(), -1, 0, 1)
child_score = rollout(child_board, next_player)

score = -child_score

print(child_score, score)

1 -1


In [144]:
stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

record = stats[(0, 1)]

record["N"] = record["N"] + 1
record["W"] = record["W"] + score
print(stats[(0, 1)])

{'N': 1, 'W': -1}


In [145]:
# 	先走指定动作，再调用 rollout。把子局面得分取负，转换为落子方的得分。
def simulate_move(board, player, move, stats):
    row, col = move
    child_board, next_player = step(board.copy(), player, row, col)

    child_score = rollout(child_board, next_player)
    score = -child_score

    record = stats[move]
    record["N"] = record["N"] + 1
    record["W"] = record["W"] + score

In [146]:
stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

simulate_move(board99, -1, (2, 2), stats)
print(stats[(2, 2)])

{'N': 1, 'W': 1}


In [147]:
stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

simulate_move(board99, -1, (2, 2), stats)
print(stats[(2, 2)])

simulate_move(board99, -1, (2, 2), stats)
print(stats[(2, 2)])

{'N': 1, 'W': 1}
{'N': 2, 'W': 2}


In [148]:
simulate_move(board99, -1, (0, 1), stats)

for move in legal_actions(board99):
    record = stats[move]
    Q = record["W"] / record["N"]
    print(move, Q)

(0, 1) -1.0
(2, 2) 1.0


In [149]:
moves = legal_actions(board99)
scores = []

for move in moves:
    record = stats[move]
    Q = record["W"] / record["N"]
    scores.append(Q)

best_index = scores.index(max(scores))

print(moves[best_index])

(2, 2)


In [150]:
stats = {
    (0, 1): {"N": 1, "W": -1},
    (2, 2): {"N": 0, "W": 0},
}

for move in legal_actions(board99):
    record = stats[move]
    if record["N"] == 0:
        simulate_move(board99, -1, move, stats)

In [151]:
print(stats[(0, 1)])
print(stats[(2, 2)])

{'N': 1, 'W': -1}
{'N': 1, 'W': 1}


In [153]:
Q = 0.5
total_N = 22
N = 2

bonus = np.sqrt(np.log(total_N) / N)

uct_score = Q + bonus

print(round(uct_score, 3))

1.743


In [154]:
total_N = 0

for move in legal_actions(board99):
    record = stats[move]
    total_N = total_N + record["N"]

print(total_N)

2


In [155]:
for move in legal_actions(board99):
    record = stats[move]
    N = record["N"]
    Q = record["W"] / N

    bonus = np.sqrt(np.log(total_N) / N)
    uct_score = Q + bonus

    print(move, round(uct_score, 3))

(0, 1) -0.167
(2, 2) 1.833


In [156]:
moves = legal_actions(board99)
scores = []

for move in moves:
    record = stats[move]
    N = record["N"]
    Q = record["W"] / N
    bonus = np.sqrt(np.log(total_N) / N)
    uct_score = Q + bonus

    scores.append(uct_score)


best_index = scores.index(max(scores))
next_move = moves[best_index]
print(next_move)

(2, 2)


In [157]:
simulate_move(board99, -1, next_move, stats)
# 补一行，调用 simulate_move


print(next_move, stats[next_move])

(2, 2) {'N': 2, 'W': 2}


In [158]:
def select_uct(board, stats):
    moves = legal_actions(board)

    for move in moves:
        record = stats[move]
        if record["N"] == 0:
            return move

In [159]:
test_stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

print(select_uct(board99, test_stats))

(0, 1)


In [165]:
test_stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

board999 = np.array(
    [
        [1, 0, 1],  # X . X
        [1, 1, -1],  # X X O
        [-1, -1, 0],  # O O .
    ]
)

move5 = select_uct(board999, test_stats)

simulate_move(board999, -1, move5, test_stats)

print(move5, test_stats[move5])

(0, 1) {'N': 1, 'W': -1}


In [ ]:
# 优先选择未模拟过的动作，其余情况根据 Q + bonus 安排下一次模拟。
def select_uct(board, stats):
    moves = legal_actions(board)
    total_N = 0

    # 优先选择未模拟过的动作
    for move in moves:
        record = stats[move]
        if record["N"] == 0:
            return move
        total_N = total_N + record["N"]

    # 所有动作都已模拟过，比较 UCT 评分
    scores = []
    for move in moves:
        record = stats[move]
        N = record["N"]
        Q = record["W"] / N
        bonus = np.sqrt(np.log(total_N) / N)
        uct_score = Q + bonus
        scores.append(uct_score)

    best_index = scores.index(max(scores))
    return moves[best_index]

In [170]:
visited_stats = {
    (0, 1): {"N": 1, "W": -1},
    (2, 2): {"N": 1, "W": 1},
}

print(select_uct(board999, visited_stats))

(2, 2)


In [ ]:
search_stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

for i in range(3):
    move=select_uct(board999,search_stats)
    simulate_move(board999,-1,move,search_stats)
    print(i + 1, move, search_stats[move])

1 (0, 1) {'N': 1, 'W': -1}
2 (2, 2) {'N': 1, 'W': 1}
3 (2, 2) {'N': 2, 'W': 2}


In [173]:
moves = legal_actions(board999)
visits = []

for move in moves:
    record = search_stats[move]
    visits.append(record["N"])
    # 补一行，把这个动作的 N 追加到 visits

best_index = visits.index(max(visits))
best_move = moves[best_index]

print(visits)
print(best_move)

[1, 2]
(2, 2)


In [ ]:
# 初始化统计，重复选择与模拟，最后返回访问次数最多的动作。
def search_action(board, player, num_simulations):
    if game_result(board, player) is not None:
        return None

    # 为当前棋盘的每个合法动作建立记录
    moves = legal_actions(board)
    stats = {}
    for move in moves:
        stats[move] = {"N": 0, "W": 0}

    # 反复选择动作、模拟并更新记录
    for i in range(num_simulations):
        move=select_uct(board,stats)
        # 调用 select_uct，把动作保存到 move
        simulate_move(board,player,move,stats)
        # 调用 simulate_move，更新 stats

    # 根据最终访问次数推荐动作
    visits = []
    for move in moves:
        record = stats[move]
        visits.append(record["N"])

    best_index = visits.index(max(visits))
    return moves[best_index]

In [175]:
before = board999.copy()

print(search_action(board999, -1, 3))  # 预期 (2, 2)
print(np.array_equal(board999, before))  # 预期 True

(2, 2)
True


In [176]:
before = board999.copy()

move = search_action(board999, -1, 3)
row, col = move
after_board, next_player = step(board999.copy(), -1, row, col)

print(move)
print(game_result(after_board, next_player))
print(np.array_equal(board999, before))

(2, 2)
-1
True
